# RAG (Retrieval-Augmented Generation)

This notebook implements the third approach in my project: a Retrieval-Augmented Generation (RAG) pipeline built specifically for Austrian tax law documents.

Unlike the inference notebook where I relied on a general model's training data, or the fine-tuning notebook where I updated the model's weights, RAG keeps the model unchanged and instead gives it access to a searchable legal document database at query time. For every question, the pipeline first retrieves the most relevant passages from the actual law texts, then passes those passages to the model as context so it can answer based on real legal source material rather than memorized approximations.

The pipeline is built in four phases: ingesting and chunking the PDF, vectorizing the chunks and loading the language model, defining the retrieval and generation logic, and running batch inference across the full 644-question dataset.

### Phase 1: Setup and Ingestion

Here I install the required libraries and load the Austrian tax law PDF into memory. The PDF contains the BAO, KStG, EStG, and UStG compiled into a single file.

Because language models have a context window limit (they can only read a fixed amount of text at once), I cannot feed the entire document in one go. Instead I split it into overlapping 500-character chunks. The overlap of 150 characters ensures that sentences cut at a chunk boundary are still fully readable in the adjacent chunk, preventing loss of meaning at the edges.

In [5]:
# Install all RAG dependencies:
# langchain: orchestrates the retrieval and generation pipeline
# sentence-transformers: converts text into numerical vectors for similarity search
# chromadb: the local vector database that stores and searches those vectors
# pypdf: reads the PDF file page by page
# bitsandbytes + accelerate: enable quantization and multi-GPU distribution for the LLM
!pip install langchain langchain-community sentence-transformers chromadb pypdf bitsandbytes accelerate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the compiled Austrian tax law PDF.
# PyPDFLoader reads each page and attaches metadata (page number, source path) to each one.
loader = PyPDFLoader("BAO, KStG 1988, EStG 1988, UStG 1994.pdf") # BAO, KStG 1988, EStG 1988 and UStG 1994, compiled in one file
pages = loader.load()

# Split the loaded pages into smaller overlapping chunks.
# chunk_size=500: each chunk is at most 500 characters long
# chunk_overlap=150: consecutive chunks share 150 characters to preserve context at boundaries
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 150)
chunks = text_splitter.split_documents(pages)


### Phase 2: Vectorization and Model Loading

Here I convert all the text chunks into numerical vectors (embeddings) and store them in a local vector database. When a question comes in later, the same embedding model converts it into a vector too, and the database finds the chunks whose vectors are mathematically closest, meaning the chunks most semantically similar to the question.

I use a multilingual sentence-transformer model because it was trained to understand semantic similarity across languages including German, which is important for Austrian legal terminology. The vectors are stored in ChromaDB, a lightweight local vector store that can search thousands of chunks in milliseconds.

I then load Gemma 1.1 2B as the language model that will generate the final answer from the retrieved context.

In [ ]:
from huggingface_hub import login

# Authenticate with HuggingFace to access gated models like Gemma
login("hugging_face_token") # hugging_face_token

# Load the multilingual embedding model.
# This model converts text into 384-dimensional vectors that capture semantic meaning.
# The multilingual variant handles German legal terminology more accurately than English-only models.
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Build the vector database from the chunks and their embeddings.
# Chroma stores both the raw text and the vectors on disk so the database persists between runs.
vector_db = Chroma.from_documents(chunks, embeddings, persist_directory = "./tax_rag_db")
print(f"RAG DB initialized with {len(chunks)} searchable tax segments.")

# Load the language model that will generate answers from retrieved context.
# Gemma 1.1 2B is a small instruction-tuned model that follows German prompts reliably.
model_name_llm = "google/gemma-1.1-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name_llm)

# Load the model without 8-bit quantization for compatibility.
# device_map='auto' distributes the model across available GPUs automatically.
model = AutoModelForCausalLM.from_pretrained(
    model_name_llm,
    device_map = "auto" # This automatically finds the GPU
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG DB initialized with 5638 searchable tax segments.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

### Phase 3: The RAG Logic

This function is the core of the pipeline. For each question it does two things: retrieval and generation.

First it searches the vector database for the 3 most semantically similar chunks to the question. These chunks are the raw legal text passages most likely to contain the answer. Then it builds a prompt that includes those passages as context and instructs the model to answer based only on what is in the context, always citing the relevant law.

The model generates a response token by token. I slice off the prompt tokens from the output so only the newly generated answer is returned, not the entire input repeated back.

In [7]:
import pandas as pd
import csv
from tqdm import tqdm

def ask_tax_rag(question):
    # Retrieve the 3 chunks from the vector database most semantically similar to the question
    docs = vector_db.similarity_search(question, k=3)

    # Concatenate the retrieved chunks into a single context string
    context = ""
    for d in docs:
        context += f"{d.page_content}\n"

    # Build the prompt using Gemma's instruction format (<start_of_turn> / <end_of_turn>).
    # The system instruction tells the model to act as an Austrian tax law expert,
    # answer concisely, and always cite the relevant law.
    # The retrieved context is injected directly into the prompt so the model
    # answers from the actual legal source text rather than from memory.
    rag_prompt = f"""<start_of_turn>user
Du bist ein Experte für das österreichische Steuerrecht. Beantworte die Frage kurz und präzise basierend auf dem KONTEXT.
Nenne immer das Gesetz.

KONTEXT:
{context}

FRAGE:
{question}<end_of_turn>
<start_of_turn>model
"""

    # Tokenize the prompt and move the input tensors to the same device as the model
    inputs = tokenizer(rag_prompt, return_tensors="pt").to(model.device)

    # Record the prompt length so I can slice it off the output later
    input_length = inputs.input_ids.shape[1]

    # Generate the answer using greedy decoding (do_sample=False for deterministic output)
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        do_sample = False,
        eos_token_id = tokenizer.eos_token_id,
    )

    # Slice off the prompt tokens, keeping only the newly generated answer tokens
    generated_tokens = outputs[0][input_length:]

    # Decode the token IDs back into a readable German string
    answer = tokenizer.decode(generated_tokens, skip_special_tokens = True)

    return answer.strip()


### Phase 4: Batch Processing

Here I run the `ask_tax_rag` function across my full dataset. I first do a 3-row test run to confirm the pipeline is producing sensible answers before committing to all 644 rows, which takes significantly longer to complete.

In [8]:
# Test run on the first 3 rows to verify the pipeline produces sensible output before running the full dataset
test_df = pd.read_csv('dataset_clean.csv')
test_sample = test_df.head(3) # Take only the first 3 rows
rag_results = []

print("Starting test run for 3 rows...")

for _, row in test_sample.iterrows():
    answer = ask_tax_rag(row['prompt'])
    print(f"\nID: {row['id']}")
    print(f"Answer: {answer}") # Check if this looks like a real sentence now!
    rag_results.append({"id": row['id'], "answer": answer})

# Preview the temporary results dataframe
output_df = pd.DataFrame(rag_results)
print("\nPreview")
print(output_df)


Starting test run for 3 rows...

ID: CORP-TAX-001
Answer: Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer ist die Summe der Bemessungsgrundlage des Vorsteuerbereignisses und der Investitionszuwachsprämie.

ID: CORP-TAX-002
Answer: Der Text gibt keine Informationen über steuerliche Konsequenzen bei einem zinslosen Darlehen an einen Gesellschafter, daher kann die Frage nicht beantwortet werden.

ID: CORP-TAX-003
Answer: Die Körperschaften, die in § 7 Abs. 3 fallenden Körperschaften sind verpflichtet, sämtliche Einkünfte den Einkünften aus Gewerbebetrieb zuzurechnen.

Preview
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Der Text gibt keine Informationen über steuerl...
2  CORP-TAX-003  Die Körperschaften, die in § 7 Abs. 3 fallende...


In [9]:
# Run the full pipeline across all 644 tax questions
test_df = pd.read_csv('dataset_clean.csv')

rag_results = []

# Loop through every question in the dataset.
# tqdm wraps the iterator to display a live progress bar showing how many rows have been processed.
for _, row in tqdm(test_df.iterrows(), total = len(test_df)):
    answer = ask_tax_rag(row['prompt'])

    # Store the question ID alongside the generated answer for later merging with ground truth
    rag_results.append({"id": row['id'], "answer": answer})

# Convert results to a DataFrame and save to CSV.
# QUOTE_ALL wraps every cell in double quotes to protect commas inside legal citations
# from being misread as column separators.
output_df = pd.DataFrame(rag_results)
output_df.to_csv("rag_results.csv", index = False, quoting = csv.QUOTE_ALL)
print("Task 3 Complete! File saved as rag_results.csv")


100%|██████████| 643/643 [22:38<00:00,  2.11s/it]

Task 3 Complete! File saved as rag_results.csv
